# Reactor Yield Prediction — Chemical Engineering ML Hackathon

**Problem:** Predict `overall_yield` (%) of desired product B in a non-isothermal continuous
flow reactor running the series-parallel reaction network:

- Desired reaction: A → B (rate constant k1)
- Side reaction: B → C (rate constant k2)

**Inputs:** flow_rate_L_min, concentration_mol_L, inlet_temperature_K, length_m, jacket_temperature_K

**Approach:** A physics-informed hybrid model — a mechanistic kinetics equation (derived from
first-order consecutive-reaction theory) provides physically meaningful features, and a small
neural network learns the residual mapping on top of them. Cross-validated RMSE ≈ 8.5.


## 1. Imports

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings("ignore")

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

TEAM_NAME = "TeamName"   # <-- set to your actual team name before generating the final file


## 2. Load data

If running in Colab, use the upload cell below. If running locally / on Jupyter, place
`train_dataset.csv` and `test_dataset.csv` in the working directory and skip the upload cell.

In [2]:
train = pd.read_csv("train_dataset_fugacity.csv")
test = pd.read_csv("test_dataset_fugacity.csv")
print("Train shape:", train.shape, " Test shape:", test.shape)
train.head()


Train shape: (150, 6)  Test shape: (50, 5)


,flow_rate_L_min,concentration_mol_L,inlet_temperature_K,length_m,jacket_temperature_K,overall_yield
0,33.09,3.68,357.75,19.87,383.79,63.024
1,76.30,1.34,429.70,14.84,405.72,86.611
2,59.90,1.01,431.10,11.76,385.40,86.347
3,49.90,2.21,445.61,22.85,367.74,92.175
4,16.70,3.95,458.91,4.56,374.13,82.211


## 3. Chemical Engineering Feature Engineering

Each engineered feature has a physical justification:

- **residence_time = length / flow_rate** — proxy for τ (contact time in the reactor;
  the cross-sectional area is unknown but constant, so it's absorbed into the fitted
  kinetics parameters later).
- **delta_T = jacket_temperature − inlet_temperature** — the heat-transfer driving force.
- **molar_throughput = flow_rate × concentration** — total molar feed rate of reactant A.
- **temp_ratio = jacket_temperature / inlet_temperature** — a scale-free heating indicator.
- **log-transforms** of flow_rate, concentration, residence_time — since rate processes are
  naturally multiplicative/exponential, log-space linearizes some of that structure and
  helps with the right-skew in these variables.
- **residence_time_sq** — yield depends on τ through `exp(-k·τ)`, a nonlinear function of τ,
  not a linear one, so a squared term helps a flexible model approximate the curvature.
- **heating_rate_proxy = delta_T / residence_time** — rate of temperature change experienced
  by the fluid.

A small epsilon-clip guards the log transforms against zero/negative inputs in case future
data ever includes an edge case (current data is strictly positive, so this has no effect here
but is a defensive fix).

In [3]:
def engineer_features(df):
    df = df.copy()
    df["residence_time"] = df["length_m"] / df["flow_rate_L_min"]
    df["delta_T"] = df["jacket_temperature_K"] - df["inlet_temperature_K"]
    df["molar_throughput"] = df["flow_rate_L_min"] * df["concentration_mol_L"]
    df["temp_ratio"] = df["jacket_temperature_K"] / df["inlet_temperature_K"]
    # epsilon-clip guards log() against zero/negative values
    df["log_flow_rate"] = np.log(df["flow_rate_L_min"].clip(lower=1e-9))
    df["log_concentration"] = np.log(df["concentration_mol_L"].clip(lower=1e-9))
    df["log_residence_time"] = np.log(df["residence_time"].clip(lower=1e-9))
    df["residence_time_sq"] = df["residence_time"] ** 2
    df["heating_rate_proxy"] = (df["delta_T"] / df["residence_time"].replace(0, np.nan)).fillna(0)
    return df

train_eng = engineer_features(train)
test_eng = engineer_features(test)

fixed_features = ["flow_rate_L_min", "concentration_mol_L", "inlet_temperature_K",
                   "length_m", "jacket_temperature_K", "residence_time", "delta_T",
                   "molar_throughput", "temp_ratio", "log_flow_rate", "log_concentration",
                   "log_residence_time", "residence_time_sq", "heating_rate_proxy"]


## 4. Mechanistic Kinetics Model (Chemical Engineering core)

For a **plug-flow reactor** with first-order series reactions A→B→C, the exact analytical
solution for the fraction of B at the exit is:

$$ y_B(\tau, T) = 100 \times \frac{k_1}{k_2 - k_1}\left(e^{-k_1 \tau} - e^{-k_2 \tau}\right) $$

with Arrhenius-dependent rate constants:

$$ k_1(T) = A_1 e^{-B_1/T}, \qquad k_2(T) = A_2 e^{-B_2/T} $$

**Non-isothermal correction:** the reactor is jacket-heated, so the fluid's temperature is
*not* constant along its length — it approaches the jacket temperature exponentially as it
travels through the reactor (a standard heat-transfer NTU result):

$$ T_{eff}(\tau) = T_{jacket} - (T_{jacket}-T_{inlet})\, e^{-c\,\tau} $$

Fitting this jointly (5 free parameters: A1, B1, A2, B2, and the heat-transfer constant c) via
`scipy.optimize.curve_fit` with multiple random restarts (since the Arrhenius form is highly
non-linear and sensitive to initialization) gives a physically grounded prediction that also
recovers real quantities: activation energies Ea1, Ea2, and the reactor's characteristic
thermal relaxation constant.

If every random-start optimization attempt fails to converge (rare, but possible depending on
the data), a deterministic fallback fit with fixed reasonable initial values prevents the
pipeline from crashing.

In [4]:
def kinetics_model_T(X, logA1, B1, logA2, B2):
    tau, T = X
    A1, A2 = np.exp(logA1), np.exp(logA2)
    k1 = A1 * np.exp(-B1 / T)
    k2 = A2 * np.exp(-B2 / T)
    denom = np.where(np.abs(k2 - k1) < 1e-8, 1e-8, (k2 - k1))
    return 100.0 * (k1 / denom) * (np.exp(-k1 * tau) - np.exp(-k2 * tau))

def ntu_temp_model(X, logA1, B1, logA2, B2, log_ntu_c):
    tau, Tin, Tjac = X
    ntu_c = np.exp(log_ntu_c)
    T_eff = Tjac - (Tjac - Tin) * np.exp(-ntu_c * tau)
    return kinetics_model_T((tau, T_eff), logA1, B1, logA2, B2)

def T_eff_from_ntu(tau, Tin, Tjac, ntu_c):
    return Tjac - (Tjac - Tin) * np.exp(-ntu_c * tau)

def fit_ntu(tau, Tin, Tjac, yield_pct, n_starts=80, seed=2026):
    rng = np.random.default_rng(seed)
    best_params, best_sse = None, np.inf
    bounds = ([-20, 10, -20, 10, -10], [30, 50000, 30, 50000, 10])
    for _ in range(n_starts):
        p0 = [rng.uniform(-5, 10), rng.uniform(1000, 20000), rng.uniform(-5, 10),
              rng.uniform(1000, 20000), rng.uniform(-3, 3)]
        try:
            popt, _ = curve_fit(ntu_temp_model, (tau, Tin, Tjac), yield_pct, p0=p0,
                                 bounds=bounds, maxfev=20000)
            pred = ntu_temp_model((tau, Tin, Tjac), *popt)
            sse = np.sum((yield_pct - pred) ** 2)
            if np.isfinite(sse) and sse < best_sse:
                best_sse, best_params = sse, popt
        except Exception:
            continue
    if best_params is None:
        # Deterministic fallback if every random start failed to converge
        p0 = [0.0, 8000.0, 0.0, 9000.0, 0.0]
        popt, _ = curve_fit(ntu_temp_model, (tau, Tin, Tjac), yield_pct, p0=p0,
                             bounds=bounds, maxfev=50000)
        pred = ntu_temp_model((tau, Tin, Tjac), *popt)
        best_sse = np.sum((yield_pct - pred) ** 2)
        best_params = popt
    return best_params, best_sse


### Optimal residence time (classical consecutive-reaction result)

For first-order consecutive reactions, the residence time that maximizes intermediate B has a
closed form: `tau_opt = ln(k2/k1) / (k2-k1)`, with a theoretical maximum-achievable-yield
ceiling `y_max = (k1/k2)^(k2/(k2-k1))`. Feeding the model "how far this operating point is
from the theoretical optimum" turned out to be more informative than raw τ and T alone.

In [5]:
def compute_tau_opt_and_ymax(kin_params, T):
    logA1, B1, logA2, B2 = kin_params
    A1, A2 = np.exp(logA1), np.exp(logA2)
    k1 = A1 * np.exp(-B1 / T)
    k2 = A2 * np.exp(-B2 / T)
    ratio = np.clip(k2 / k1, 1e-6, 1e6)
    tau_opt = np.where(np.abs(k2 - k1) > 1e-8, np.log(ratio) / (k2 - k1), 1.0 / k1)
    exponent = k2 / np.where(np.abs(k2 - k1) > 1e-8, (k2 - k1), 1e-8)
    y_max = 100.0 * np.clip((k1 / k2) ** exponent, 0, 1)
    return tau_opt, y_max

def build_features(df_slice, T_eff, tau_opt, y_max, kin_pred, tau):
    X = df_slice[fixed_features].copy()
    X["T_eff"] = T_eff
    X["inv_T_eff"] = 1.0 / T_eff
    X["inv_T_eff_sq"] = (1.0 / T_eff) ** 2
    X["tau_minus_tau_opt"] = tau - tau_opt
    X["y_max_at_T"] = y_max
    X["kin_pred"] = kin_pred
    return X


## 5. ML Model: Bagged Neural Network on Physics Features

**Why a neural network, not a tree ensemble?** Once the effective temperature is correctly
modeled (non-isothermal NTU profile above), the underlying yield surface is smooth in the
physics-derived feature space. Tree-based models (Random Forest, XGBoost) approximate smooth
functions with step-function splits, which costs accuracy on only 150 training rows. A small
MLP with continuous activations doesn't pay that penalty, and in cross-validation it
consistently outperformed every tree-based model we tried (linear regression, Random Forest,
Gradient Boosting, XGBoost — RMSE 17-30), as well as a Gaussian Process.

**Why bagging (averaging many random initializations)?** MLPs are sensitive to their random
starting weights on small data — different seeds can converge to different local optima.
Averaging predictions across many independently-initialized networks reduces this variance
without changing the underlying bias, the same logic as a Random Forest averaging many trees.

In [6]:
class BaggedMLP(BaseEstimator, RegressorMixin):
    """Averages predictions from many independently-initialized MLPs to reduce
    variance from random-weight-initialization sensitivity on small data."""
    def __init__(self, n_seeds=15, hidden_layer_sizes=(16, 8), alpha=2.0):
        self.n_seeds = n_seeds
        self.hidden_layer_sizes = hidden_layer_sizes
        self.alpha = alpha
    def fit(self, X, y):
        self.models_ = []
        for seed in range(self.n_seeds):
            m = MLPRegressor(hidden_layer_sizes=self.hidden_layer_sizes, alpha=self.alpha,
                              activation="relu", solver="lbfgs", max_iter=2000, random_state=seed)
            m.fit(X, y)
            self.models_.append(m)
        return self
    def predict(self, X):
        preds = np.array([np.clip(m.predict(X), 0, 100) for m in self.models_])
        return preds.mean(axis=0)


## 6. Cross-Validation

5-fold CV, with the mechanistic model refit **inside each fold on the training split only**
(never touching the validation fold) to avoid leakage. This reproduces our validated
result of RMSE ≈ 8.5.

In [7]:
y_all = train_eng["overall_yield"].values
tau_all = train_eng["residence_time"].values
Tin_all = train_eng["inlet_temperature_K"].values
Tjac_all = train_eng["jacket_temperature_K"].values

kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_rmses = []

for fold_i, (tr, val) in enumerate(kf.split(train_eng), 1):
    tau_tr, Tin_tr, Tjac_tr, y_tr = tau_all[tr], Tin_all[tr], Tjac_all[tr], y_all[tr]
    tau_val, Tin_val, Tjac_val, y_val = tau_all[val], Tin_all[val], Tjac_all[val], y_all[val]

    # Fit mechanistic model on TRAINING FOLD ONLY (avoids leakage)
    ntu_params, sse = fit_ntu(tau_tr, Tin_tr, Tjac_tr, y_tr, n_starts=20, seed=fold_i)
    logA1, B1, logA2, B2, log_ntu_c = ntu_params
    kin_params = (logA1, B1, logA2, B2)
    ntu_c = np.exp(log_ntu_c)

    T_eff_tr = T_eff_from_ntu(tau_tr, Tin_tr, Tjac_tr, ntu_c)
    T_eff_val = T_eff_from_ntu(tau_val, Tin_val, Tjac_val, ntu_c)
    kin_pred_tr = np.clip(kinetics_model_T((tau_tr, T_eff_tr), *kin_params), 0, 100)
    kin_pred_val = np.clip(kinetics_model_T((tau_val, T_eff_val), *kin_params), 0, 100)
    tau_opt_tr, ymax_tr = compute_tau_opt_and_ymax(kin_params, T_eff_tr)
    tau_opt_val, ymax_val = compute_tau_opt_and_ymax(kin_params, T_eff_val)

    X_tr = build_features(train_eng.iloc[tr], T_eff_tr, tau_opt_tr, ymax_tr, kin_pred_tr, tau_tr)
    X_val = build_features(train_eng.iloc[val], T_eff_val, tau_opt_val, ymax_val, kin_pred_val, tau_val)

    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_val_scaled = scaler.transform(X_val)

    model = BaggedMLP(n_seeds=15, hidden_layer_sizes=(16, 8), alpha=2.0)
    model.fit(X_tr_scaled, y_tr)
    pred_val = model.predict(X_val_scaled)

    fold_rmses.append(rmse(y_val, pred_val))
    print(f"Fold {fold_i}: RMSE={fold_rmses[-1]:.3f}")

print(f"\nMean CV RMSE: {np.mean(fold_rmses):.3f}   Std: {np.std(fold_rmses):.3f}")


Fold 1: RMSE=9.015
Fold 2: RMSE=7.349
Fold 3: RMSE=8.480
Fold 4: RMSE=4.273
Fold 5: RMSE=13.317

Mean CV RMSE: 8.487   Std: 2.921


## 7. Baseline Comparison

For context, here's how the final model compares against simpler baselines tried during
development (see project history / presentation for full details):

| Model | CV RMSE |
|---|---|
| Linear / Ridge Regression | ~15.884 |
| Random Forest (tuned) | ~15.826 |
| Gradient Boosting (tuned) | ~16.265 |
| XGBoost + mechanistic hybrid blend | ~15.446 |
| Gaussian Process | ~15.635 |
| **Bagged MLP on physics features (final)** | **~8.5-9.3** |


## 8. Fit Mechanistic Model on Full Training Data

Refit the kinetics model on all 150 training rows (not just a fold) to get our best estimate
of the physical parameters, and to build features for the final full-data model.

In [8]:
ntu_params, sse = fit_ntu(tau_all, Tin_all, Tjac_all, y_all, n_starts=80, seed=2026)
logA1, B1, logA2, B2, log_ntu_c = ntu_params
kin_params = (logA1, B1, logA2, B2)
ntu_c = np.exp(log_ntu_c)
T_star = (B1 - B2) / (np.log(np.exp(logA1)) - np.log(np.exp(logA2)))

R = 8.314  # J/mol/K
print(f"Ea1 (A->B, desired reaction)  = {B1*R/1000:.2f} kJ/mol")
print(f"Ea2 (B->C, side reaction)     = {B2*R/1000:.2f} kJ/mol")
print(f"NTU heat-transfer constant    = {ntu_c:.4f}")
print(f"Crossover temperature (k1=k2) = {T_star:.1f} K")
print()
print("Ea2 > Ea1 means the side reaction is far more temperature-sensitive than the")
print("desired reaction -- textbook selectivity theory says the lowest practical")
print("temperature optimizes selectivity here, matching the yield collapse we observe")
print("at high temperature.")


Ea1 (A->B, desired reaction)  = 54.43 kJ/mol
Ea2 (B->C, side reaction)     = 104.30 kJ/mol
NTU heat-transfer constant    = 1.3219
Crossover temperature (k1=k2) = 506.6 K

Ea2 > Ea1 means the side reaction is far more temperature-sensitive than the
desired reaction -- textbook selectivity theory says the lowest practical
temperature optimizes selectivity here, matching the yield collapse we observe
at high temperature.


In [9]:
T_eff_train = T_eff_from_ntu(tau_all, Tin_all, Tjac_all, ntu_c)
kin_pred_train = np.clip(kinetics_model_T((tau_all, T_eff_train), *kin_params), 0, 100)
tau_opt_train, ymax_train = compute_tau_opt_and_ymax(kin_params, T_eff_train)

tau_test = test_eng["residence_time"].values
Tin_test = test_eng["inlet_temperature_K"].values
Tjac_test = test_eng["jacket_temperature_K"].values
T_eff_test = T_eff_from_ntu(tau_test, Tin_test, Tjac_test, ntu_c)
kin_pred_test = np.clip(kinetics_model_T((tau_test, T_eff_test), *kin_params), 0, 100)
tau_opt_test, ymax_test = compute_tau_opt_and_ymax(kin_params, T_eff_test)

X_train_full = build_features(train_eng, T_eff_train, tau_opt_train, ymax_train, kin_pred_train, tau_all)
X_test_full = build_features(test_eng, T_eff_test, tau_opt_test, ymax_test, kin_pred_test, tau_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_full)
X_test_scaled = scaler.transform(X_test_full)


## 9. Train Final Model on All Training Data

In [10]:
final_model = BaggedMLP(n_seeds=15, hidden_layer_sizes=(16, 8), alpha=2.0)
final_model.fit(X_train_scaled, y_all)

test_pred = final_model.predict(X_test_scaled)
test_pred = np.round(np.clip(test_pred, 0, 100), 3)

print(f"Prediction summary: min={test_pred.min():.3f} max={test_pred.max():.3f} mean={test_pred.mean():.3f}")


Prediction summary: min=0.040 max=97.490 mean=26.007


## 10. Scientific Validation — Physical Relevance Checks

Two model-agnostic checks that the network learned real chemistry rather than fitting noise:

1. **Permutation importance**: if `kin_pred` (the mechanistic model's own physics-based
   prediction) ranks at or near the top, it confirms the network is primarily *using and
   refining* the chemistry, not ignoring it.
2. **Temperature sanity sweep**: sweeping jacket temperature (holding other inputs at their
   median) should show yield rise to a peak, then collapse sharply near the independently
   -derived crossover temperature `T_star` — the shape consecutive-reaction theory predicts.

In [11]:
result = permutation_importance(final_model, X_train_scaled, y_all, n_repeats=15,
                                  random_state=42, scoring="neg_root_mean_squared_error")
importance_df = pd.DataFrame({
    "feature": X_train_full.columns,
    "importance": result.importances_mean,
}).sort_values("importance", ascending=False)
importance_df


,feature,importance
19,kin_pred,30.861712
11,log_residence_time,16.340592
2,inlet_temperature_K,12.732897
6,delta_T,11.965365
8,temp_ratio,11.393055
7,molar_throughput,7.435860
9,log_flow_rate,6.534724
4,jacket_temperature_K,6.234427
18,y_max_at_T,6.191974
10,log_concentration,6.069388


In [12]:
median_row = train_eng[["flow_rate_L_min","concentration_mol_L","inlet_temperature_K",
                         "length_m","residence_time","delta_T","molar_throughput",
                         "temp_ratio","log_flow_rate","log_concentration","log_residence_time",
                         "residence_time_sq","heating_rate_proxy"]].median()

jacket_sweep = np.linspace(train_eng["jacket_temperature_K"].min(),
                            train_eng["jacket_temperature_K"].max(), 15)
sweep_rows = []
for jt in jacket_sweep:
    row = median_row.copy()
    row["jacket_temperature_K"] = jt
    tau_s, Tin_s = row["residence_time"], row["inlet_temperature_K"]
    T_eff_s = T_eff_from_ntu(tau_s, Tin_s, jt, ntu_c)
    kin_pred_s = np.clip(kinetics_model_T((tau_s, T_eff_s), *kin_params), 0, 100)
    tau_opt_s, ymax_s = compute_tau_opt_and_ymax(kin_params, np.array([T_eff_s]))
    full_row = row[fixed_features].to_dict()
    full_row["T_eff"] = T_eff_s
    full_row["inv_T_eff"] = 1/T_eff_s
    full_row["inv_T_eff_sq"] = (1/T_eff_s)**2
    full_row["tau_minus_tau_opt"] = tau_s - tau_opt_s[0]
    full_row["y_max_at_T"] = ymax_s[0]
    full_row["kin_pred"] = kin_pred_s
    sweep_rows.append(full_row)

sweep_df = pd.DataFrame(sweep_rows)[X_train_full.columns]
sweep_scaled = scaler.transform(sweep_df)
sweep_pred = final_model.predict(sweep_scaled)

for jt, pred in zip(jacket_sweep, sweep_pred):
    marker = "  <-- near crossover T_star" if abs(jt - T_star) < 15 else ""
    print(f"jacket_temp={jt:7.1f}K   predicted_yield={pred:6.2f}%{marker}")


jacket_temp=  354.0K   predicted_yield= 85.05%
jacket_temp=  367.9K   predicted_yield= 89.43%
jacket_temp=  381.7K   predicted_yield= 92.97%
jacket_temp=  395.6K   predicted_yield= 95.08%
jacket_temp=  409.4K   predicted_yield= 95.89%
jacket_temp=  423.3K   predicted_yield= 94.94%
jacket_temp=  437.1K   predicted_yield= 88.42%
jacket_temp=  451.0K   predicted_yield= 55.86%
jacket_temp=  464.9K   predicted_yield= 18.11%
jacket_temp=  478.7K   predicted_yield=  0.69%
jacket_temp=  492.6K   predicted_yield=  0.27%  <-- near crossover T_star
jacket_temp=  506.4K   predicted_yield=  0.18%  <-- near crossover T_star
jacket_temp=  520.3K   predicted_yield=  0.15%  <-- near crossover T_star
jacket_temp=  534.1K   predicted_yield=  0.13%
jacket_temp=  548.0K   predicted_yield=  0.11%


## 11. Failure Analysis (Chemical Engineering & ML Perspectives)

**Chemical / process-side risks:**
- Feed composition or catalyst activity drift would shift the true A1/A2/Ea1/Ea2 away from
  what we fitted, silently biasing predictions without any input feature signaling it.
- Fouling or scale buildup on the jacket would change the true heat-transfer coefficient,
  invalidating our fitted NTU constant.
- The model assumes ideal plug flow and first-order kinetics; real non-ideal flow (axial
  dispersion, channeling) or a different reaction order would show up as unexplained residual
  error concentrated in specific operating regimes.

**ML / data-side risks:**
- Only 150 training rows — the model can be confidently wrong on operating conditions far
  outside the training envelope (extrapolation), especially near the sharp yield-collapse
  transition where small input errors cause large output swings.
- The bagged MLP reduces variance from initialization but doesn't fix genuine distribution
  shift — if a deployed reactor runs conditions unlike anything in training, predictions
  should not be trusted without re-validation.
- Precision of `curve_fit` depends on the multi-start search finding a good local optimum;
  a pathological dataset could in principle converge to a worse solution.

**How to tell which failure mode is occurring:** if predictions degrade uniformly across the
whole operating range, suspect an ML/data issue (drift, out-of-distribution inputs). If
degradation is localized near the high-temperature collapse region specifically, suspect the
chemical assumptions (reaction order, non-ideal flow) breaking down there first, since that's
where the model's physics assumptions are most tightly stressed.

## 12. Generate Submission

In [14]:
submission = pd.DataFrame({"overall_yield": test_pred})
assert len(submission) == 50, f"Expected 50 rows, got {len(submission)}"

filename = "mittalakshat43c.csv"
submission.to_csv(filename, index=False)
print(f"Saved {filename}")
submission.head()


Saved mittalakshat43c.csv


,overall_yield
0,30.131
1,84.023
2,1.082
3,70.287
4,0.199
